# 01 · Exploring PTB-XL

Before training anything we need to know what the model will see. This notebook
answers four questions, each of which fixes a later design choice:

1. **How imbalanced are the classes?** Decides the headline metric.
2. **How often do labels co-occur?** Decides the output layer.
3. **Do the official folds keep the class balance?** Decides whether val/test numbers can be trusted.
4. **What does preprocessing do to a real signal?** Decides the filter band.

All logic lives in `fedecg.data`; this notebook only calls it and explains the
results. The same tables are written to `results/tables/` by
`scripts/explore_data.py`.

Prerequisite: `uv run python scripts/download_data.py`.

In [ ]:
%matplotlib inline
from fedecg.config import load_config
from fedecg.data.constants import SUPERCLASSES
from fedecg.data.preprocess import LeadStandardizer, filter_from_config
from fedecg.data.ptbxl import (
    attach_labels,
    load_diagnostic_map,
    load_metadata,
    load_signals,
    split_by_folds,
)
from fedecg.data.stats import (
    cooccurrence,
    example_ids,
    label_cardinality,
    label_combinations,
    label_distribution,
    records_by,
)
from fedecg.viz import plot_class_examples, plot_ecg

config = load_config("default.yaml")
data_cfg = config["data"]
FS = data_cfg["sampling_rate_hz"]

## From SCP codes to five superclasses

Each record carries a dict of SCP-ECG statements, e.g. `{'IMI': 80.0, 'SR': 0.0}`,
where the number is the cardiologist's likelihood. Only *diagnostic* statements
map to a superclass; rhythm (`SR`) and form (`PVC`) statements do not.

Following the PTB-XL benchmark, a diagnostic code counts **regardless of its
likelihood**, and records left with no superclass are dropped.

In [ ]:
raw = load_metadata()
diagnostic_map = load_diagnostic_map()
meta = attach_labels(raw, diagnostic_map)
print(
    f"{len(raw)} records, {len(meta)} with at least one superclass ({len(raw) - len(meta)} dropped)"
)
meta[["patient_id", "scp_codes", "strat_fold", *SUPERCLASSES]].head()

## 1. Class balance → macro AUROC

If one class dominates, accuracy rewards a model that ignores the rare ones.
Macro AUROC averages a threshold-free score over classes with equal weight, so
`HYP` counts as much as `NORM`.

In [ ]:
split = split_by_folds(
    meta,
    train_folds=data_cfg["train_folds"],
    val_fold=data_cfg["val_fold"],
    test_fold=data_cfg["test_fold"],
)
distribution = label_distribution(meta, split)
distribution

## 2. Co-occurrence → independent sigmoids

If records routinely carry several superclasses, a softmax (which forces
probabilities to compete and sum to one) is the wrong output. The model instead
predicts five independent probabilities, trained with binary cross-entropy.

In [ ]:
display(label_cardinality(meta).to_frame())
display(label_combinations(meta).to_frame())
cooccurrence(meta)

## 3. Folds preserve the balance

`strat_fold` was built by the dataset authors to stratify labels *and* keep
every patient within one fold, so there is no patient leakage between train and
test. The `*_pct` columns above should be close across `train`, `val` and
`test`.

In [ ]:
ax = distribution[["train_pct", "val_pct", "test_pct"]].plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("records with label (%)");

## Natural heterogeneity: site and device

Phase 5 partitions data into "hospitals" using the recording `site` and
`device`. These tables show how many records each group holds and how different
their class mix is — the raw material for non-IID experiments.

In [ ]:
display(records_by(meta, "device").head(10))
records_by(meta, "site").head(10)

## What each class looks like

One record per superclass (preferring records with that label alone), lead II,
band-passed. MI typically shows pathological Q waves or ST changes, CD a widened
QRS, HYP increased amplitudes.

In [ ]:
examples = example_ids(meta)
names = list(examples)
signals = load_signals(meta.loc[[examples[n] for n in names]], sampling_rate=FS)
filtered = filter_from_config(signals, data_cfg["preprocess"], fs=FS)
plot_class_examples(filtered, names, fs=FS);

## 4. Preprocessing

- **Band-pass 0.5–40 Hz**, zero-phase: removes baseline wander (breathing,
  electrode drift) and high-frequency noise without shifting waves in time.
- **Per-lead standardization** with statistics fitted on the training split
  only. Fitting on everything would leak test information; in the federated
  phases each hospital can only fit on its own data.

Raw (grey) against filtered (red):

In [ ]:
plot_ecg(signals[0], overlay=filtered[0], fs=FS, title=f"{names[0]}: raw vs. band-passed");

In [ ]:
standardizer = LeadStandardizer().fit(filtered)
standardized = standardizer.transform(filtered)
print("per-lead mean:", standardized.mean(axis=(0, 2)).round(3))
print("per-lead std: ", standardized.std(axis=(0, 2)).round(3))

## Takeaways

- Imbalanced classes → **macro AUROC** as the headline metric, per-class F1 alongside.
- Frequent co-occurrence → **multi-label** output with independent sigmoids.
- Official folds keep class balance and separate patients → used as-is.
- Filtering and standardization live in `fedecg.data.preprocess` and are driven
  by `data.preprocess` in the config.